# Ytools-Story — YouTube Faceless Automation

Loop footage + overlay (watermark, particles, karaoke subtitle) + TTS narration.
Bawa script sendiri (wajib) atau pakai LLM (openai/anthropic, butuh API key). Bilingual ID/EN.

**Cara pakai:** Run semua cell top-to-bottom (Shift+Enter).
Upload footage di **Cell 4**, atur opsi di **Cell 5**, upload script di **Cell 6**, lalu lihat hasil di **Cell 7-8**.


In [ ]:
#@title 1. Install dependencies (~1 min)
!pip install -q edge-tts ffmpeg-python imageio-ffmpeg pillow numpy pyyaml

import os, sys, shutil, subprocess
print('python', sys.version.split()[0])

REPO = 'https://github.com/fawaz333888/Ytools-Story.git'
DEST = '/content/Ytools-Story'

def _have_ytools(path):
    return os.path.isfile(os.path.join(path, 'ytools', '__init__.py'))

if not _have_ytools(DEST):
    ok = subprocess.run(['git', 'clone', '-q', REPO, DEST]).returncode == 0
    if not ok or not _have_ytools(DEST):
        # fallback: copy from the cwd this notebook was launched from
        src = os.getcwd()
        if _have_ytools(src) and os.path.abspath(src) != os.path.abspath(DEST):
            shutil.copytree(src, DEST, dirs_exist_ok=True)
            print('clone failed - copied local copy from', src)
        else:
            print('ERROR: clone gagal dan tidak ada copy lokal. Push repo ke GitHub dulu.')

os.chdir(DEST)
sys.path.insert(0, DEST)

from ytools import __version__
print('ytools', __version__)

In [ ]:
#@title 2. Cek environment (GPU / ffmpeg)
import subprocess
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                         capture_output=True, text=True).stdout or 'no GPU')
except Exception as e:
    print('nvidia-smi error:', e)

from ytools.render.ffutil import FFRunner
ff = FFRunner()
print('ffmpeg:', ff.version())
print('ffprobe:', ff.ffprobe or 'tidak ada (fallback parse ffmpeg -i)')
print('nvenc flag:', ff.supports_nvenc())

In [ ]:
#@title 3. Mount Google Drive (sekali, untuk simpan hasil)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Ytools-Story'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive siap:', DRIVE_ROOT)

In [ ]:
#@title 4. Upload footage video (minimal 30 detik)
from google.colab import files
import shutil, os

FOOTAGE = 'footage.mp4'
if not os.path.isfile(FOOTAGE):
    uploaded = files.upload()
    if uploaded:
        name = list(uploaded.keys())[0]
        shutil.move(name, FOOTAGE)
        print('saved as', FOOTAGE)
    else:
        print('tidak ada file diupload')

if os.path.isfile(FOOTAGE):
    info = ff.probe(FOOTAGE)
    print(f"footage: {info['width']}x{info['height']} {info['duration']:.1f}s fps={info['fps']:.1f} audio={info['has_audio']}")
    if info['duration'] < 30:
        print('WARNING: footage < 30 detik. Loop terlalu repetitif untuk YouTube.')

In [ ]:
#@title 5. Konfigurasi
#@markdown --- **Story** ---
niche = 'horror' #@param ['horror','motivation','education','drama','custom']
language = 'id' #@param ['id','en']
provider = 'manual' #@param ['manual','openai','anthropic']
topic = '' #@param {type:'string'}
length_minutes = 3 #@param {type:'number'}  # hanya untuk LLM; manual mengikuti panjang script
seed = None #@param {type:'raw'}

#@markdown --- **Narration (TTS)** ---
voice = '' #@param {type:'string'}  # kosong = auto by language
tts_rate = '+0%' #@param {type:'string'}

#@markdown --- **Video** ---
width = 1280 #@param {type:'integer'}
height = 720 #@param {type:'integer'}
fps = 30 #@param {type:'integer'}
motion = 'slow_drift' #@param ['none','slow_drift','slow_zoom']

#@markdown --- **Overlays** ---
particle_style = 'dust' #@param ['dust','snow','sparkle','fireflies','embers','fog']
particle_density = 120 #@param {type:'integer'}
watermark_text = '@YtoolsChannel' #@param {type:'string'}
watermark_style = 'badge' #@param ['badge','plain','logo']
watermark_position = 'top-right' #@param ['top-left','top-right','bottom-left','bottom-right','center']
subtitle_style = 'karaoke' #@param ['karaoke','simple']

#@markdown --- **Output** ---
output_name = 'video.mp4' #@param {type:'string'}

#@markdown --- **LLM key (hanya jika provider != manual)** ---
llm_model = 'gpt-4o-mini' #@param {type:'string'}
llm_base_url = '' #@param {type:'string'}  # kosong = endpoint resmi; isi untuk OpenAI-compatible (OpenRouter/Groq/vLLM)
llm_api_key = '' #@param {type:'string'}  # kosong = pakai Colab secret; JANGAN isi kalau notebook dibagikan
os.environ.pop('OPENAI_API_KEY', None)
if provider == 'openai':
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = llm_api_key or userdata.get('OPENAI_API_KEY', '')
elif provider == 'anthropic':
    from google.colab import userdata
    os.environ.pop('ANTHROPIC_API_KEY', None)
    os.environ['ANTHROPIC_API_KEY'] = llm_api_key or userdata.get('ANTHROPIC_API_KEY', '')

from ytools.config import Config
cfg = Config()
cfg.set('story.niche', niche); cfg.set('story.language', language)
cfg.set('story.provider', provider); cfg.set('story.topic', topic)
cfg.set('story.length_minutes', length_minutes)
cfg.set('story.model', llm_model or None)
cfg.set('story.base_url', llm_base_url or None)
if provider == 'anthropic':
    cfg.set('story.api_key_env', 'ANTHROPIC_API_KEY')
cfg.set('story.seed', seed)
cfg.set('tts.voice', voice or None); cfg.set('tts.rate', tts_rate)
cfg.set('video.width', width); cfg.set('video.height', height)
cfg.set('video.fps', fps); cfg.set('video.motion', motion)
cfg.set('overlays.particles.style', particle_style)
cfg.set('overlays.particles.density', particle_density)
cfg.set('overlays.watermark.text', watermark_text)
cfg.set('overlays.watermark.style', watermark_style)
cfg.set('overlays.watermark.position', watermark_position)
cfg.set('overlays.subtitle.style', subtitle_style)
cfg.set('output.encoder', 'auto')
problems = cfg.validate()
print('config OK' if not problems else 'config problems:', problems)

In [ ]:
#@title 6. RUN pipeline (story -> tts -> overlays -> render)
from ytools.pipeline import Pipeline

workdir = '/content/ytools_work/run'
pipe = Pipeline(workdir, cfg, ff=ff)

script_arg = None
if provider == 'manual':
    import os
    SCRIPT = '/content/script.txt'
    if not os.path.isfile(SCRIPT):
        from google.colab import files
        up = files.upload()
        assert up, 'provider=manual wajib upload file script .txt'
        name = list(up.keys())[0]
        open(SCRIPT, 'wb').write(up[name])
    script_arg = SCRIPT

art = pipe.run(footage=FOOTAGE, script_path=script_arg)

from IPython.display import HTML
print('duration:', art.duration, 'words:', len(art.words))

In [ ]:
#@title 7. Preview hasil
from IPython.display import HTML, display
from base64 import b64encode

out = art.result.path
mp4 = open(out, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
display(HTML(f'<video width=640 controls><source src="{data_url}"></video>'))
print('size MB:', len(mp4) / 1e6)

In [ ]:
#@title 8. Simpan hasil ke Drive
import shutil, json, datetime

run_dir = os.path.join(DRIVE_ROOT, datetime.date.today().isoformat())
os.makedirs(run_dir, exist_ok=True)

shutil.copy(art.result.path, os.path.join(run_dir, output_name))
shutil.copy(art.script_path, os.path.join(run_dir, output_name.replace('.mp4', '_script.txt')))

meta = {
    'niche': niche,
    'language': language,
    'provider': provider,
    'length_minutes': length_minutes,
    'duration': art.duration,
    'footage': FOOTAGE,
    'created': datetime.datetime.now().isoformat(),
}
meta_path = os.path.join(run_dir, output_name.replace('.mp4', '_meta.json'))
with open(meta_path, 'w', encoding='utf-8') as fh:
    json.dump(meta, fh, ensure_ascii=False, indent=2)

print('saved to', run_dir)
for f in sorted(os.listdir(run_dir)):
    print('  ', f)

## Catatan

- **Session Colab free** maksimal ~12 jam, idle disconnect ~90 menit. Hasil otomatis tersimpan ke Drive (cell 8) — aman berhenti kapan saja.
- **Reuse**: cell 6 menggunakan cache — rerun cepat kalau footage/particles/watermark sudah ada.
- **Story**: provider `manual` wajib bawa script sendiri. Untuk skrip panjang (>5 menit) gunakan `openai`/`anthropic`.
- **Footage < 30 detik** akan terlalu repetitif; YouTube demote konten loop pendek berulang.
